# Week 4 — Pandas Introduction: Thursday Exercises
## Phase 2a Python | PORA Academy Cohort 7

**Dataset:** `olist_order_items_dataset.csv` (112,650 rows × 7 columns)

**Instructions:**
- Run the data setup cell first — it loads `items` into memory
- Every question has a verified expected output — confirm yours matches before moving on
- Use DeepSeek only after genuinely attempting the task yourself (5 minutes minimum)
- All placeholder code is in comments — write your solution on the blank lines below each comment
- Thursday's assignment is at the bottom of this notebook — complete it before next session

**What you will practise today:**
- Loading a CSV with numeric columns and reading `.describe()` output
- Adding calculated columns (`total_cost`, `price_tier`)
- Sorting with `.sort_values()` and finding top-N with `.nlargest()` / `.nsmallest()`
- Aggregating with `.groupby()` and ranking sellers by revenue

## Setup — Run This First

In [41]:
import pandas as pd
from google.colab import drive
import os

drive.mount('/content/drive')

olist_items_path = '/content/olist_order_items_dataset.csv'
olist_orders_path = '/content/olist_orders_dataset.csv'

items_df = pd.read_csv(olist_items_path)
orders_df = pd.read_csv(olist_orders_path)

print(f"Loaded {len(items_df):,} items with {items_df.shape[1]} columns")
print(f"Loaded {len(orders_df):,} orders with {orders_df.shape[1]} columns")
# expected: Loaded 112,650 items with 7 columns

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 112,650 items with 7 columns
Loaded 99,441 orders with 8 columns


## Question 1 — Inspect the Dataset

Print the **shape** of `items` and use `.describe()` on the `price` and `freight_value` columns.

Expected shape: **(112650, 7)**

From `.describe()`, confirm:
- `price` mean ≈ 120.65, max = 6735.00, min = 0.85
- `freight_value` mean ≈ 19.99

In [7]:
# Q1: Print shape and describe price + freight_value
print(items.shape)
print()
# describe price + freight_value
print(items[['price', 'freight_value']].describe().round(2))

(112650, 7)

           price  freight_value
count  112650.00      112650.00
mean      120.65          19.99
std       183.63          15.81
min         0.85           0.00
25%        39.90          13.08
50%        74.99          16.26
75%       134.90          21.15
max      6735.00         409.68


## Question 2 — Add Total Cost Column

Add a new column `total_cost` to `items` equal to `price + freight_value`.

Then print:
- The **mean** of `total_cost` (rounded to 2 decimal places) — Expected: **140.64**
- The **sum** of `total_cost` formatted as `R$X,XXX,XXX.XX` — Expected: **R$15,843,553.24**

In [9]:
# Q2: Add total_cost column, print mean and sum
items['total_cost'] = items['price'] + items['freight_value']
print(items['total_cost'].mean().round(2))
print()
print(f"R${items['total_cost'].sum()}")


140.64

R$15843553.24


## Question 3 — Classify Items by Price Tier

Add a `price_tier` column using `.apply()` with these rules:
- **Premium**: price ≥ R$500
- **Standard**: price ≥ R$100 (and < R$500)
- **Economy**: price < R$100

Print the count of each tier using `.value_counts()`.

Expected:
```
Economy     72165
Standard    37246
Premium      3239
```

In [10]:
# Q3: Add price_tier column and print value_counts
items['price_tier'] = items['price'].apply(
    lambda x: 'Premium' if x >= 500 else 'Standard' if x >= 100 else 'Economy'
)
print(items['price_tier'].value_counts())


price_tier
Economy     72165
Standard    37246
Premium      3239
Name: count, dtype: int64


## Question 4 — Top and Bottom Items by Price

Use `.nlargest()` to find the **5 most expensive items** (show `product_id` and `price` columns).

Then use `.nsmallest()` to find the **5 cheapest items**.

Expected: most expensive item price = **R$6,735.00**, cheapest = **R$0.85**

In [15]:
# Q4a: 5 most expensive items
top5_price = items.nlargest(5, 'price')[['product_id', 'price', 'freight_value']]
print(top5_price)
print()
# Q4b: 5 cheapest items
bottom5_price = items.nsmallest(5, 'price')[['product_id', 'price']]
print(bottom5_price)


                              product_id   price  freight_value
3556    489ae2aa008f021502940f251d4cce7f  6735.0         194.31
112233  69c590f7ffc7bf8db97190b6cb6ed62e  6729.0         193.21
107841  1bdf5e6731585cf01aa8169c7028d6ad  6499.0         227.66
74336   a6492cc69376c469ab6f61d8f44de961  4799.0         151.34
11249   c3ed642d592594bb648ff4a04cee2747  4690.0          74.34

                             product_id  price
27652  8a3254bee785a526d548a81a9bc3c9be   0.85
48625  8a3254bee785a526d548a81a9bc3c9be   0.85
87081  8a3254bee785a526d548a81a9bc3c9be   0.85
57297  270516a3f41dc035aa87d220228f844c   1.20
57298  05b515fdc76e888aada3c6d66c201dff   1.20


## Question 5 — Top 5 Sellers by Revenue

Group `items` by `seller_id`, sum the `price` column, and find the top 5 sellers by total revenue.

Expected: the #1 seller's total revenue = **R$229,472.63**

In [16]:
# Q5: Group by seller_id, sum price, find top 5
seller_revenue = items.groupby('seller_id')['price'].sum()
print(seller_revenue.nlargest(5))


seller_id
4869f7a5dfa277a7dca6462dcf3b52b2    229472.63
53243585a1d6dc2643021fd1853d8905    222776.05
4a3ca9315b744ce9f8e9374361493884    200472.92
fa1c13f2614d7b5c4749cbc52fecda94    194042.03
7c67e1448b00f6e969d365cea6b010ab    187923.89
Name: price, dtype: float64


## Question 6 — Total Revenue and Freight

Print:
1. Total revenue (sum of `price` column) — Expected: **R$13,591,643.70**
2. Total freight collected (sum of `freight_value`) — Expected: **R$2,251,909.54**

Format both as `R$X,XXX,XXX.XX` using an f-string.

In [19]:
# Q6: Print total revenue and total freight
total_revenue = items['price'].sum()
total_freight = items['freight_value'].sum()
print(f"Total revenue: R${total_revenue:,.2f}")
print()
print(f"Total freight: R${total_freight:,.2f}")

Total revenue: R$13,591,643.70

Total freight: R$2,251,909.54


## Group Exercise — Items Analysis

These are the same tasks from the in-class group exercise. Complete them here to confirm your understanding.

Using `items` (already loaded above):

In [31]:
# Group Task 1: Total revenue (sum of price)
# Expected: R$13,591,643.70
total_revenue = items['price'].sum()
total_freight = items['freight_value'].sum()
print(f"Total revenue: R${total_revenue:,.2f}")
print()

# Group Task 2: Total freight collected
# Expected: R$2,251,909.54
print(f"Total freight: R${total_freight:,.2f}")
print()

# Group Task 3: Add 'revenue_share' = price / total_revenue * 100
# (each item's % contribution to total revenue)
items['revenue_share'] = items['price'] / total_revenue * 100
print(items[['price', 'revenue_share']].head())
print()

# Group Task 4: How many items are in the Premium tier (price >= R$500)?
# Expected: 3,239 items
premium_items = items[items['price_tier'] == 'Premium']
print(f"Number of premium items: {len(premium_items)}")
print()

# Group Task 5: Which order has the most items?
# Hint: groupby order_id, count order_item_id, nlargest(1)
# Expected: max items in one order = 21
order_item_counts = items.groupby('order_id')['order_item_id'].count()
print(f"Order with most items: {order_item_counts.nlargest(1)}")
print()

# Group Task 6: Average freight as a % of price
# Expected: ~32%
average_freight_share = items['freight_value'].sum() / items['price'].sum() * 100
print(f"Average freight as a % of price: {average_freight_share:.2f}%")

Total revenue: R$13,591,643.70

Total freight: R$2,251,909.54

    price  revenue_share
0   58.90       0.000433
1  239.90       0.001765
2  199.00       0.001464
3   12.99       0.000096
4  199.90       0.001471

Number of premium items: 3239

Order with most items: order_id
8272b63d03f5f79c56e9e4120aec44ef    21
Name: order_item_id, dtype: int64

Average freight as a % of price: 16.57%


## Weekly Assignment 4

**Due before next session. Submit `week4_assignment.ipynb`.**

1. Load both `olist_orders_dataset.csv` and `olist_order_items_dataset.csv`.
   Print `.shape` and `.isnull().sum()` for each.

2. From the orders DataFrame:
   - How many unique order statuses exist? *(Expected: 8)*
   - What % of orders are NOT yet delivered (status ≠ 'delivered')? *(Expected: 3.0%)*
   - Filter to only orders with status 'canceled' or 'unavailable'. How many rows? *(Expected: 1,234)*

3. From the items DataFrame:
   - Add a `total_cost` column (price + freight_value)
   - Add a `price_tier` column: Premium (≥500), Standard (≥100), Economy (<100)
   - What is the count of each tier? *(Expected: Economy=72,165 / Standard=37,246 / Premium=3,239)*

4. Find the top 5 sellers by total revenue (sum of price, grouped by seller_id).
   What is the revenue of the #1 seller? *(Expected: R$229,472.63)*

1. Load both `olist_orders_dataset.csv` and `olist_order_items_dataset.csv`.
   Print `.shape` and `.isnull().sum()` for each.

In [42]:
import pandas as pd
from google.colab import drive
import os

drive.mount('/content/drive')

olist_items_path = '/content/olist_order_items_dataset.csv'
olist_orders_path = '/content/olist_orders_dataset.csv'

items_df = pd.read_csv(olist_items_path)
orders_df = pd.read_csv(olist_orders_path)

print('Shape of items_df:', items_df.shape)
print('Null values in items_df:\n', items_df.isnull().sum())
print('\nShape of orders_df:', orders_df.shape)
print('Null values in orders_df:\n', orders_df.isnull().sum())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape of items_df: (112650, 7)
Null values in items_df:
 order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Shape of orders_df: (99441, 8)
Null values in orders_df:
 order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


2. From the orders DataFrame:
   - How many unique order statuses exist? *(Expected: 8)*
   - What % of orders are NOT yet delivered (status ≠ 'delivered')? *(Expected: 3.0%)*
   - Filter to only orders with status 'canceled' or 'unavailable'. How many rows? *(Expected: 1,234)*

In [43]:
# 2. From the orders DataFrame:
#    - How many unique order statuses exist? *(Expected: 8)
unique_order_statuses = orders_df['order_status'].nunique()
print(f"Number of unique order statuses: {unique_order_statuses}")
print()
#    - What % of orders are NOT yet delivered (status ≠ 'delivered')? *(Expected: 3.0%)
not_delivered_orders = orders_df[orders_df['order_status'] != 'delivered']
percentage_not_delivered = (len(not_delivered_orders) / len(orders_df)) * 100
print(f"Percentage of orders not yet delivered: {percentage_not_delivered:.1f}%")
print()
#    - Filter to only orders with status 'canceled' or 'unavailable'. How many rows? *(Expected: 1,234)
canceled_or_unavailable_orders = orders_df[orders_df['order_status'].isin(['canceled', 'unavailable'])]
print(f"Number of orders with 'canceled' or 'unavailable' status: {len(canceled_or_unavailable_orders)}")

Number of unique order statuses: 8

Percentage of orders not yet delivered: 3.0%

Number of orders with 'canceled' or 'unavailable' status: 1234


3. From the items DataFrame:
   - Add a `total_cost` column (price + freight_value)
   - Add a `price_tier` column: Premium (≥500), Standard (≥100), Economy (<100)
   - What is the count of each tier? *(Expected: Economy=72,165 / Standard=37,246 / Premium=3,239)*

In [44]:
# Add a total_cost column (price + freight_value)
items_df['total_cost'] = items_df['price'] + items_df['freight_value']
print(items_df[['price', 'freight_value', 'total_cost']].head())
print()

#Add a price_tier column: Premium (≥500), Standard (≥100), Economy (<100)
items_df['price_tier'] = items_df['price'].apply(
    lambda x: 'Premium' if x >= 500 else 'Standard' if x >= 100 else 'Economy'
)
print(items_df[['price', 'price_tier']].head())
# What is the count of each tier? (Expected: Economy=72,165 / Standard=37,246 / Premium=3,239)
print(items_df['price_tier'].value_counts())

    price  freight_value  total_cost
0   58.90          13.29       72.19
1  239.90          19.93      259.83
2  199.00          17.87      216.87
3   12.99          12.79       25.78
4  199.90          18.14      218.04

    price price_tier
0   58.90    Economy
1  239.90   Standard
2  199.00   Standard
3   12.99    Economy
4  199.90   Standard
price_tier
Economy     72165
Standard    37246
Premium      3239
Name: count, dtype: int64


4. Find the top 5 sellers by total revenue (sum of price, grouped by seller_id).
   What is the revenue of the #1 seller? *(Expected: R$229,472.63)*

In [48]:
# Q4: Find the top 5 sellers by total revenue
seller_revenue_q4 = items_df.groupby('seller_id')['price'].sum()
top_5_sellers = seller_revenue_q4.nlargest(5)
print(top_5_sellers)
print()
# What is the revenue of the #1 seller?
seller_revenue = items_df.groupby('seller_id')['price'].sum()
print(f"Revenue of the #1 seller: R${top_5_sellers.iloc[0]:,.2f}")
# Expected: R$229,472.63

seller_id
4869f7a5dfa277a7dca6462dcf3b52b2    229472.63
53243585a1d6dc2643021fd1853d8905    222776.05
4a3ca9315b744ce9f8e9374361493884    200472.92
fa1c13f2614d7b5c4749cbc52fecda94    194042.03
7c67e1448b00f6e969d365cea6b010ab    187923.89
Name: price, dtype: float64

Revenue of the #1 seller: R$229,472.63
